# 서울에서 5억으로 아파트를 살 수 있을까?
2025년 서울 아파트 실거래 분석 | 데이터 시각화 뉴스 콘텐츠 제작

**실행 방법** Colab에서 **런타임 → 모두 실행**을 선택하세요. 저장소를 내려받아 실행하면 `data` 폴더의 CSV를 읽고, Colab에서는 GitHub에서 자동으로 내려받습니다. 실행 결과는 실행 환경의 `generated` 폴더에 저장됩니다.

출처: [서울시 부동산 실거래가 정보](https://data.seoul.go.kr/dataList/OA-21275/S/1/datasetView.do), 2026.11.05 수집.
2025년 계약, 아파트, 취소·분양권·입주권 제외. 가격은 실거래금액이며 세금 등 부대비용 제외.
현재 매물이나 고유 주택 수가 아닌 **거래 건수** 기준입니다. 원래 과제의 데이터 선택지 밖이므로 제출 전 자유 주제 허용 여부를 확인하세요.


In [ ]:
from datetime import datetime
from pathlib import Path
from shutil import copyfileobj
from urllib.request import urlopen

filename = 'seoul-apartments-2025.csv'
data_paths = [
    Path('data') / filename,
    Path('../data') / filename,
    Path(filename),
]
DATA = next((path for path in data_paths if path.is_file()), data_paths[0])
if not DATA.is_file():
    url = 'https://raw.githubusercontent.com/YuNi-100/data-visualization/main/data/seoul-apartments-2025.csv'
    DATA.parent.mkdir(parents=True, exist_ok=True)
    temporary = DATA.with_suffix('.download')
    try:
        with urlopen(url, timeout=60) as response, temporary.open('wb') as file:
            copyfileobj(response, file)
        temporary.replace(DATA)
    finally:
        temporary.unlink(missing_ok=True)

OUT = Path('generated') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT.mkdir(parents=True)
print('저장 위치', OUT.resolve())


## 1. 데이터와 한글 글꼴 준비

In [ ]:
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

font_paths = [
    Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'),
    Path('C:/Windows/Fonts/malgun.ttf'),
]
font_path = next((path for path in font_paths if path.is_file()), None)
if font_path is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], check=True)
    font_path = font_paths[0]
font_manager.fontManager.addfont(str(font_path))
plt.rcParams.update({
    'font.family': font_manager.FontProperties(fname=font_path).get_name(),
    'axes.unicode_minus': False,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
df = pd.read_csv(DATA, encoding='utf-8-sig', parse_dates=['계약일'])
assert len(df) == 75129, '원래 준비한 데이터와 건수가 다릅니다.'
assert df['계약일'].dt.year.eq(2025).all()
assert df['거래금액_만원'].gt(0).all()
assert df['오억이하'].eq(df['거래금액_만원'].le(50000).astype(int)).all()
assert df['오억이하'].sum() == 8017
display(df.head())
print(f'전체 {len(df):,}건 / 5억 이하 {df["오억이하"].sum():,}건 / 비중 {df["오억이하"].mean():.2%}')


In [ ]:
monthly = df.groupby('계약월').agg(전체거래건수=('오억이하', 'size'), 오억이하건수=('오억이하', 'sum'))
regional = df.groupby('자치구').agg(전체거래건수=('오억이하', 'size'), 오억이하건수=('오억이하', 'sum'))
for table in [monthly, regional]:
    table['오억이하비중_pct'] = table['오억이하건수'] / table['전체거래건수'] * 100
monthly.to_csv(OUT / '월별_요약.csv', encoding='utf-8-sig')
regional.to_csv(OUT / '자치구별_요약.csv', encoding='utf-8-sig')
display(regional.sort_values('오억이하건수', ascending=False))
SOURCE = '출처: 서울열린데이터광장 | 2025년 계약, 취소·분양권·입주권 제외 | 2026.11.05 수집'
def finish(fig, filename):
    fig.text(0.07, 0.025, SOURCE, fontsize=9, color='#555555')
    fig.tight_layout(rect=(0, 0.08, 1, 0.98))
    fig.savefig(OUT / filename, dpi=200, facecolor='white')
    plt.show()
    plt.close(fig)


## 2. 선그래프 — 월별 5억 이하 거래 비중
비중의 변화는 그달에 거래된 주택 구성의 영향도 받습니다. 같은 집의 가격 변동률을 뜻하지 않습니다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(monthly.index, monthly['오억이하비중_pct'], marker='o', color='#176B87', linewidth=2.5)
for month, value in monthly['오억이하비중_pct'].items():
    ax.annotate(f'{value:.1f}%', (month, value), xytext=(0, 9), textcoords='offset points', ha='center', fontsize=9)
ax.set(title='2025년 서울 아파트, 5억 이하 거래 비중은 월별로 얼마나 달랐나?', xlabel='계약월', ylabel='전체 아파트 거래 중 5억 이하 비중 (%)')
ax.set_ylim(0, monthly['오억이하비중_pct'].max() * 1.25)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=.2)
finish(fig, '01_선그래프_월별비중.png')


## 3. 막대그래프 — 자치구별 5억 이하 거래 건수
모든 자치구를 포함합니다. 거래 건수와 해당 구 전체 거래에서 차지하는 비중은 구분하세요.

In [ ]:
plot_data = regional.sort_values('오억이하건수')
fig, ax = plt.subplots(figsize=(10, 11))
bars = ax.barh(plot_data.index, plot_data['오억이하건수'], color='#176B87')
ax.bar_label(bars, labels=[f'{v:,}' for v in plot_data['오억이하건수']], padding=4, fontsize=9)
ax.set(title='2025년 서울, 5억 이하 아파트 거래는 어디에 많았나?', xlabel='5억 이하 거래 건수 (건)', ylabel='자치구')
ax.set_xlim(0, plot_data['오억이하건수'].max() * 1.18)
ax.grid(axis='x', alpha=.2)
finish(fig, '02_막대그래프_자치구별건수.png')


## 4. 히스토그램 — 전체 거래가격 분포
1억 원 간격으로 모든 거래가격을 포함합니다. 5억 이하의 정확한 건수는 원자료 조건으로 계산하며 구간 막대에서 추정하지 않습니다.

In [ ]:
prices = df['거래금액_억원']
bins = np.arange(0, np.floor(prices.max()) + 2, 1)
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(prices, bins=bins, color='#176B87', edgecolor='white')
ax.axvline(5, color='#D35D36', linestyle='--', linewidth=2, label='5억 원 기준')
ax.set(title='2025년 서울 아파트, 실제로 거래된 가격의 분포', xlabel='거래금액 (억 원, 1억 원 간격)', ylabel='거래 건수 (건)')
ax.set_xlim(0, bins[-1])
ax.set_ylim(bottom=0)
ax.legend()
ax.grid(axis='y', alpha=.2)
finish(fig, '03_히스토그램_거래가격분포.png')
assert np.histogram(prices, bins=bins)[0].sum() == len(df)


## 5. 차트 설명과 이야기 3문장 자동 생성

In [ ]:
peak = monthly['오억이하비중_pct'].idxmax()
low = monthly['오억이하비중_pct'].idxmin()
top = regional['오억이하건수'].idxmax()
story = [
    f'2025년 서울 아파트 거래 {len(df):,}건 가운데 5억 원 이하 거래는 {df["오억이하"].sum():,}건으로 {df["오억이하"].mean():.2%}였다.',
    f'5억 이하 거래가 가장 많은 자치구는 {top}로 {regional.loc[top, "오억이하건수"]:,}건이며, 구 전체 거래의 {regional.loc[top, "오억이하비중_pct"]:.2f}%였다.',
    f'도봉구는 5억 이하 거래가 {regional.loc["도봉구", "오억이하건수"]:,}건이며, 구 전체 거래의 {regional.loc["도봉구", "오억이하비중_pct"]:.2f}%였다.'
]
captions = [
    f'선그래프: 5억 이하 거래 비중은 {peak}에 {monthly.loc[peak, "오억이하비중_pct"]:.2f}%로 가장 높고, {low}에 {monthly.loc[low, "오억이하비중_pct"]:.2f}%로 가장 낮았다.',
    f'막대그래프: 5억 이하 거래 건수는 {top}가 {regional.loc[top, "오억이하건수"]:,}건으로 가장 많았다.',
    f'히스토그램: 전체 거래의 {df["오억이하"].mean():.2%}가 5억 원 이하였고, 거래가격 중앙값은 {prices.median():.2f}억 원이었다.'
]
text = '\n'.join(captions + [''] + [f'{i}. {s}' for i, s in enumerate(story, 1)])
print(text)
(OUT / '차트설명과_이야기3문장.txt').write_text(text, encoding='utf-8')
print('차트 3장과 요약 자료 저장 완료:', OUT)


## 6. 문제 2 Power BI 대시보드

숫자 카드는 전체 거래 건수, 5억 원 이하 거래 건수, 5억 원 이하 거래 비중을 보여 줍니다. 월별 선그래프는 **거래 건수**를, 앞의 문제 1 선그래프는 **거래 비중**을 나타냅니다. 자치구별 막대그래프, 가격 구분 도넛 차트, 자치구별 비교표를 함께 배치했습니다.

| 자치구 슬라이서 | 전체 거래 | 5억 원 이하 | 비중 |
| --- | ---: | ---: | ---: |
| 모두 | 75,129건 | 8,017건 | 10.67% |
| 강동구 | 4,926건 | 182건 | 3.69% |

[프로젝트 README](../README.md)에서 필터 전후 캡처와 Power BI 실행 방법을 확인할 수 있습니다.

### 대시보드에서 확인한 사실

1. 2025년 서울 아파트 거래 75,129건 중 5억 원 이하 거래는 8,017건으로 10.67%였다.
2. 서울 자치구 중 5억 원 이하 거래가 가장 많았던 곳은 노원구로, 1,790건이었다.
3. 강동구를 선택하면 전체 4,926건 중 182건이 5억 원 이하로, 비중은 3.69%였다.
